# Day 9: Advanced Data Wrangling & High-Performance Pandas
> **Programme:** C-DAC PGCP-AI (Post Graduate Certificate Programme in AI)  
> **Course:** Data Analytics  
> **Author:** abs6187 (`23f2000876@ds.study.iitm.ac.in`)  
> **Date:** October 13, 2026  

---

### 🎯 Learning Objectives:
1. **MultiIndex DataFrame & Hierarchical Slicing:** Construct multi-tiered row and column indices (`from_product`, `from_tuples`), perform precise multidimensional indexing with `pd.IndexSlice` and `.xs()`, reshape dimensions with `stack()` / `unstack()`, and aggregate across index levels.
2. **Vectorised Strings & Regular Expressions:** Leverage the compiled C `.str` accessor to eliminate slow Python loops, clean unstructured text, perform regex extraction with named capture groups (`extract`, `extractall`), and standardize noisy formats with regex backreferences.
3. **Transform Pipeline:** Implement declarative, reproducible data processing pipelines using method chaining and `df.pipe()`, master grouped transformations (`groupby().transform()`) for group-specific Z-scores and imputation, and enforce schema contracts.
4. **Memory Optimisation & Data Type Downcasting:** Profile DataFrame memory footprint at byte-level resolution (`memory_usage(deep=True)`), safely downcast integer and floating-point bitwidths, convert low-cardinality strings to `category`, and build an automated production optimizer delivering $80\%+$ RAM reduction.
5. **High-Performance Query (`df.query` & `pd.eval`):** Master concise query syntax with `@variable` bindings and backtick escaped column names, optimize compound arithmetic with `pd.eval()` by eliminating temporary array allocations, and benchmark multi-threaded NumExpr engine acceleration across 1,000,000 rows.

---

## 📑 Table of Contents
1. [Module 1: MultiIndex DataFrame and Hierarchical Slicing](#1-multiindex-dataframe-and-hierarchical-slicing)
   - 1.1 MultiIndex Construction (Tuples, Arrays, Cartesian Product)
   - 1.2 Multi-Level Column Hierarchies
   - 1.3 Hierarchical Slicing with `.loc` and `pd.IndexSlice`
   - 1.4 Cross-Sectional Retrieval (`.xs()`)
   - 1.5 Structural Reshaping (`stack()`, `unstack()`, `swaplevel()`)
   - 1.6 Multi-Level Aggregations (`groupby(level=...)`)
2. [Module 2: Vectorised Strings and Regular Expressions](#2-vectorised-strings-and-regular-expressions)
   - 2.1 The `.str` Accessor vs. Python Loops
   - 2.2 Text Normalization, Splitting & Expanding (`split(expand=True)`)
   - 2.3 Regular Expression Pattern Matching & Extraction with Named Groups
   - 2.4 Multiple Pattern Extraction (`extractall()`)
   - 2.5 Regex Substitution & Backreferences (`replace(regex=True)`)
3. [Module 3: Transform Pipeline](#3-transform-pipeline)
   - 3.1 Declarative Pipelines via Method Chaining and `df.pipe()`
   - 3.2 Modular Functional Steps: Imputation, Outlier Handling, Feature Engineering
   - 3.3 Grouped Transformations: `groupby().transform()` vs `apply()`
   - 3.4 Group-Specific Z-Score Scaling & Group Median Imputation
   - 3.5 End-to-End Composable Pipeline Execution
4. [Module 4: Memory Optimisation and Data Type Downcasting](#4-memory-optimisation-and-data-type-downcasting)
   - 4.1 Internal Memory Architecture & Deep Memory Profiling
   - 4.2 Numeric Downcasting (`int64` $\to$ `int8`/`int16`, `float64` $\to$ `float32`)
   - 4.3 High vs Low Cardinality: Categorical Encoding (`category`)
   - 4.4 Automated Production Downcasting Function (`optimize_dataframe_memory()`)
   - 4.5 Before & After Memory Benchmarks (RAM Reduction $> 80\%$)
5. [Module 5: High-Performance Query (`df.query` & `pd.eval`)](#5-high-performance-query)
   - 5.1 Query Syntax & Expression Elegance (`df.query()`)
   - 5.2 Dynamic Variable Binding via `@variable`
   - 5.3 Handling Columns with Spaces Using Backticks (`` `Col Name` ``)
   - 5.4 Arithmetic Optimization with `df.eval()` & `pd.eval()`
   - 5.5 Benchmark: Boolean Indexing vs `query(python)` vs `query(numexpr)` on 1,000,000 Rows
6. [Summary & Architectural Scorecard](#6-summary--architectural-scorecard)

---
## 1. MultiIndex DataFrame and Hierarchical Slicing

### Theoretical Foundation:
A **MultiIndex** (hierarchical index) allows representation of higher-dimensional data structures ($N > 2$) inside standard two-dimensional tabular DataFrames. Instead of flat denormalized tables or complex nested dictionaries, MultiIndex maintains an ordered hierarchy of levels:

$$\text{Index Tuple} = (L_0, L_1, L_2, \dots, L_{k-1})$$

Each level possesses its own name and unique labels. This provides:
1. **$O(1)$ Hash / B-Tree Lookups:** Rapid querying across any hierarchical combination.
2. **Compact Tabular Storage:** Eliminates redundant string repetition across rows.
3. **Natural Aggregation Boundaries:** Grouping or rolling summaries along any level axis without reshaping.

In [1]:
import pandas as pd
import numpy as np
import time
import re

# Set random seed for reproducibility
rng = np.random.default_rng(seed=42)

# Set Pandas display formatting
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)

print("=" * 80)
print("MODULE 1.1: CONSTRUCTING MULTIINDEX (ROWS & COLUMNS)")
print("=" * 80)

# 1. Define MultiIndex Levels for Global Enterprise Sales Data
regions = ['North America', 'EMEA', 'APAC']
countries_map = {
    'North America': ['USA', 'Canada'],
    'EMEA': ['Germany', 'UK'],
    'APAC': ['Japan', 'India']
}
years = [2025, 2026]
quarters = ['Q1', 'Q2']

# Build row tuples matching realistic geographical hierarchy
row_tuples = []
for region, countries in countries_map.items():
    for country in countries:
        for year in years:
            for q in quarters:
                row_tuples.append((region, country, year, q))

row_index = pd.MultiIndex.from_tuples(
    row_tuples,
    names=['Region', 'Country', 'Year', 'Quarter']
)

# 2. Define Multi-Level Hierarchical Columns (Categories -> Specific Metrics)
col_tuples = [
    ('Financial', 'Revenue_M'),
    ('Financial', 'OpEx_M'),
    ('Financial', 'EBITDA_M'),
    ('Operations', 'Headcount'),
    ('Operations', 'CSAT_Score')
]

col_index = pd.MultiIndex.from_tuples(
    col_tuples,
    names=['Metric_Type', 'Metric']
)

# 3. Generate Realistic Corporate Data Matrix
n_rows = len(row_index)
revenue = np.round(rng.uniform(50.0, 300.0, size=n_rows), 2)
opex = np.round(revenue * rng.uniform(0.55, 0.75, size=n_rows), 2)
ebitda = np.round(revenue - opex, 2)
headcount = rng.integers(120, 2500, size=n_rows)
csat = np.round(rng.uniform(78.0, 98.5, size=n_rows), 1)

data_matrix = np.column_stack([revenue, opex, ebitda, headcount, csat])

# Instantiate Hierarchical DataFrame
df_enterprise = pd.DataFrame(data_matrix, index=row_index, columns=col_index)
# Ensure operations headcount is integer-represented
df_enterprise[('Operations', 'Headcount')] = df_enterprise[('Operations', 'Headcount')].astype(int)

# Sort the MultiIndex for optimal slicing performance
df_enterprise.sort_index(inplace=True)

print(f"Hierarchical DataFrame Shape: {df_enterprise.shape}")
print(f"Row Index Levels ({df_enterprise.index.nlevels}): {df_enterprise.index.names}")
print(f"Column Index Levels ({df_enterprise.columns.nlevels}): {df_enterprise.columns.names}")
print("\nFirst 8 Rows of Hierarchical Enterprise DataFrame:")
display(df_enterprise.head(8))

MODULE 1.1: CONSTRUCTING MULTIINDEX (ROWS & COLUMNS)
Hierarchical DataFrame Shape: (24, 5)
Row Index Levels (4): ['Region', 'Country', 'Year', 'Quarter']
Column Index Levels (2): ['Metric_Type', 'Metric']

First 8 Rows of Hierarchical Enterprise DataFrame:


Metric_Type                 Financial                  Operations           
Metric                      Revenue_M  OpEx_M EBITDA_M  Headcount CSAT_Score
Region Country Year Quarter                                                 
APAC   India   2025 Q1         239.52  171.60    67.92        208       91.6
                    Q2         138.63   98.56    40.07        452       86.3
               2026 Q1         292.67  183.65   109.02        704       94.7
                    Q2         273.28  166.06   107.22        392       81.4
       Japan   2025 Q1         188.65  120.25    68.40       1442       84.0
                    Q2          65.95   47.26    18.69       1212       91.6
               2026 Q1         256.91  177.28    79.63       1323       89.4
                    Q2         207.92  127.35    80.57       1473       94.1

### 1.2 Hierarchical Slicing with `.loc` & `pd.IndexSlice`

#### The Slicing Trap:
In a flat DataFrame, slicing uses `df.loc[row_slice, col_slice]`. In a MultiIndex:
- If you slice by tuple: `df.loc[('North America', 'USA')]` it slices the leading levels.
- To slice **inner levels** (e.g. all `Q1` data across all regions and countries), a plain `:` syntax inside tuples raises a `SyntaxError`.
- **Solution:** Use `pd.IndexSlice` (`idx = pd.IndexSlice`), which constructs standard Python slice objects for arbitrary levels.

> **CRITICAL RULE:** A MultiIndex **must be sorted** (`df.sort_index()`) prior to range slicing, or Pandas will raise an `UnsortedIndexError` or trigger slow unindexed fallbacks!

In [2]:
print("=" * 80)
print("MODULE 1.2: HIERARCHICAL SLICING WITH pd.IndexSlice")
print("=" * 80)

idx = pd.IndexSlice

# Example A: Slice outer level directly
print("--- [A] Slicing Outer Level: Region == 'North America' ---")
na_slice = df_enterprise.loc['North America']
display(na_slice.head(4))

# Example B: Precise Multi-Level Slice via IndexSlice
# Filter: Any Region, Any Country, Year 2026, Quarter 'Q1'
# Columns: Financial Category (Revenue_M and EBITDA_M only)
print("\n--- [B] IndexSlice: All Regions & Countries for 2026 Q1 Financial Metrics ---")
q1_2026_financials = df_enterprise.loc[
    idx[:, :, 2026, 'Q1'],
    idx['Financial', ['Revenue_M', 'EBITDA_M']]
]
display(q1_2026_financials)

# Example C: Range Slicing on Hierarchical Index
# Filter: Regions from 'APAC' to 'EMEA', Year 2025
print("\n--- [C] Range Slice: Regions 'APAC' to 'EMEA', Year 2025 ---")
range_slice = df_enterprise.loc[
    idx['APAC':'EMEA', :, 2025, :],
    :
]
display(range_slice.head(6))

MODULE 1.2: HIERARCHICAL SLICING WITH pd.IndexSlice
--- [A] Slicing Outer Level: Region == 'North America' ---


Metric_Type          Financial                  Operations           
Metric               Revenue_M  OpEx_M EBITDA_M  Headcount CSAT_Score
Country Year Quarter                                                 
Canada  2025 Q1          73.54   42.72    30.82       2101       91.0
             Q2         293.91  201.80    92.11        595       89.3
        2026 Q1         240.28  167.94    72.34       2035       89.5
             Q2         246.52  183.29    63.23        137       84.2


--- [B] IndexSlice: All Regions & Countries for 2026 Q1 Financial Metrics ---


Metric_Type                        Financial         
Metric                             Revenue_M EBITDA_M
Region        Country Year Quarter                   
APAC          India   2026 Q1         292.67   109.02
              Japan   2026 Q1         256.91    79.63
EMEA          Germany 2026 Q1         142.70    50.81
              UK      2026 Q1         160.85    65.08
North America Canada  2026 Q1         240.28    72.34
              USA     2026 Q1         264.65    94.39


--- [C] Range Slice: Regions 'APAC' to 'EMEA', Year 2025 ---


Metric_Type                 Financial                  Operations           
Metric                      Revenue_M  OpEx_M EBITDA_M  Headcount CSAT_Score
Region Country Year Quarter                                                 
APAC   India   2025 Q1         239.52  171.60    67.92        208       91.6
                    Q2         138.63   98.56    40.07        452       86.3
       Japan   2025 Q1         188.65  120.25    68.40       1442       84.0
                    Q2          65.95   47.26    18.69       1212       91.6
EMEA   Germany 2025 Q1          82.03   50.46    31.57       2016       78.6
                    Q2         162.60  101.48    61.12       1992       87.0

### 1.3 Cross-Sectional Retrieval with `.xs()`

The `.xs()` method provides an expressive, readable API for extracting cross-sections at specific inner levels:
- `df.xs(key, level, axis=0, drop_level=True)`
- Setting `drop_level=False` preserves the full hierarchical coordinate metadata.
- Setting `axis=1` retrieves column-level slices seamlessly.

In [3]:
print("=" * 80)
print("MODULE 1.3: CROSS-SECTIONAL RETRIEVAL VIA .xs()")
print("=" * 80)

# Cross-section across rows: Extract all Q2 data across all regions/countries/years
print("--- [A] Row Cross-Section: All Q2 Records (drop_level=False) ---")
q2_cross_section = df_enterprise.xs(key='Q2', level='Quarter', drop_level=False)
display(q2_cross_section.head(6))

# Cross-section across columns: Extract all Operations metrics across the entire hierarchy
print("\n--- [B] Column Cross-Section: Operations Metrics (axis=1) ---")
ops_cross_section = df_enterprise.xs(key='Operations', axis=1, level='Metric_Type')
display(ops_cross_section.head(6))

MODULE 1.3: CROSS-SECTIONAL RETRIEVAL VIA .xs()
--- [A] Row Cross-Section: All Q2 Records (drop_level=False) ---


Metric_Type                 Financial                  Operations           
Metric                      Revenue_M  OpEx_M EBITDA_M  Headcount CSAT_Score
Region Country Year Quarter                                                 
APAC   India   2025 Q2         138.63   98.56    40.07        452       86.3
               2026 Q2         273.28  166.06   107.22        392       81.4
       Japan   2025 Q2          65.95   47.26    18.69       1212       91.6
               2026 Q2         207.92  127.35    80.57       1473       94.1
EMEA   Germany 2025 Q2         162.60  101.48    61.12       1992       87.0
               2026 Q2         281.69  165.60   116.09       1702       86.4


--- [B] Column Cross-Section: Operations Metrics (axis=1) ---


Metric                       Headcount  CSAT_Score
Region Country Year Quarter                       
APAC   India   2025 Q1             208        91.6
                    Q2             452        86.3
               2026 Q1             704        94.7
                    Q2             392        81.4
       Japan   2025 Q1            1442        84.0
                    Q2            1212        91.6

### 1.4 Structural Reshaping & Aggregations

Hierarchical DataFrames can pivot and aggregate fluidly:
- **`unstack(level)`:** Moves an index level into column headers (pivoting from tall to wide).
- **`stack(level)`:** Moves a column level into the row index (pivoting from wide to tall).
- **`swaplevel(i, j)`:** Swaps the order of two index levels.
- **`groupby(level=...)`:** Groups and aggregates along specific hierarchical levels without dropping coordinates.

In [4]:
print("=" * 80)
print("MODULE 1.4: RESHAPING (STACK/UNSTACK) & LEVEL-WISE AGGREGATION")
print("=" * 80)

# 1. Unstack the 'Quarter' level to widen columns
print("--- [A] Unstacking 'Quarter' to Columns for Revenue_M ---")
revenue_series = df_enterprise[('Financial', 'Revenue_M')]
unstacked_revenue = revenue_series.unstack(level='Quarter')
display(unstacked_revenue.head(6))

# 2. Level Reordering / Swapping
print("\n--- [B] Swapping Levels: ('Quarter', 'Year') to Outer Positions ---")
swapped_df = df_enterprise.swaplevel('Region', 'Quarter').sort_index()
display(swapped_df.head(4))

# 3. Aggregation across Hierarchical Levels
# Calculate Total Revenue and Average CSAT grouped by (Region, Year)
print("\n--- [C] Hierarchical Aggregation: Groupby Level (Region, Year) ---")
level_agg = df_enterprise.groupby(level=['Region', 'Year']).agg({
    ('Financial', 'Revenue_M'): 'sum',
    ('Financial', 'EBITDA_M'): 'sum',
    ('Operations', 'CSAT_Score'): 'mean'
})
level_agg.columns = ['Total_Revenue_M', 'Total_EBITDA_M', 'Avg_CSAT']
display(level_agg)

MODULE 1.4: RESHAPING (STACK/UNSTACK) & LEVEL-WISE AGGREGATION
--- [A] Unstacking 'Quarter' to Columns for Revenue_M ---


Quarter                  Q1      Q2
Region Country Year                
APAC   India   2025  239.52  138.63
               2026  292.67  273.28
       Japan   2025  188.65   65.95
               2026  256.91  207.92
EMEA   Germany 2025   82.03  162.60
               2026  142.70  281.69


--- [B] Swapping Levels: ('Quarter', 'Year') to Outer Positions ---


Metric_Type                        Financial                  Operations           
Metric                             Revenue_M  OpEx_M EBITDA_M  Headcount CSAT_Score
Quarter Country Year Region                                                        
Q1      Canada  2025 North America     73.54   42.72    30.82       2101       91.0
                2026 North America    240.28  167.94    72.34       2035       89.5
        Germany 2025 EMEA              82.03   50.46    31.57       2016       78.6
                2026 EMEA             142.70   91.89    50.81       1977       82.4


--- [C] Hierarchical Aggregation: Groupby Level (Region, Year) ---


Total_Revenue_M  Total_EBITDA_M  Avg_CSAT
Region        Year                                           
APAC          2025           632.75          195.08    88.375
              2026          1030.78          376.44    89.900
EMEA          2025           711.29          272.87    85.975
              2026           692.05          265.74    82.950
North America 2025           770.66          260.25    89.925
              2026           975.79          328.95    89.250

---
## 2. Vectorised Strings and Regular Expressions

### Why the `.str` Accessor is Mandatory:
In raw Python, applying string functions using `df['col'].apply(lambda x: ...)` suffers from three critical bottlenecks:
1. **Python Virtual Machine Overhead:** Iterates through Python objects sequentially, incurring interpreter call overhead on every row.
2. **Missing Value Failures:** A `None` or `np.nan` triggers `TypeError: expected string or bytes-like object`, requiring ugly guard clauses.
3. **Memory Unpacking:** Scalar Python string objects are unboxed from memory arrays.

The Pandas `.str` accessor delegates processing to compiled C/Cython algorithms, natively propagates `NaN` without exceptions, and supports full Regular Expressions (`re`) out of the box.

In [5]:
print("=" * 80)
print("MODULE 2.1: MESSY UNSTRUCTURED TELEMETRY & AUDIT DATA")
print("=" * 80)

# Realistic messy enterprise audit dataset
raw_customer_records = [
    {
        'raw_id': 'USR-101',
        'raw_name': '   dr. alexander j. o\'connor, ph.d.   ',
        'raw_contact': '+1 (800) 555-0199 ext 402',
        'raw_txn': 'TXN_2026-NA-USA-ORD#88412_USD$14,500.50_STATUS:COMPLETED_FLAG:AUDIT',
        'feedback': 'Amazing experience! Loved the #Speed and #Reliability of delivery. #TopTier'
    },
    {
        'raw_id': 'USR-102',
        'raw_name': '  ms. priya sharma-patel  ',
        'raw_contact': '+91 98765-43210',
        'raw_txn': 'TXN_2025-AP-IND-ORD#19042_INR₹4,20,000.00_STATUS:PENDING_FLAG:NONE',
        'feedback': 'Great support team. Solved #Billing issue within 10 minutes! #SupportWin'
    },
    {
        'raw_id': 'USR-103',
        'raw_name': 'herr DR. KLAUS VON SCHMIDT',
        'raw_contact': '+49 (030) 123456',
        'raw_txn': 'TXN_2026-EM-DEU-ORD#99231_EUR€8,950.00_STATUS:COMPLETED_FLAG:EXPRESS',
        'feedback': 'System was down yesterday. #Downtime #Frustrated with response speed.'
    },
    {
        'raw_id': 'USR-104',
        'raw_name': '  SARAH CONNOR ',
        'raw_contact': '020 7946 0912',
        'raw_txn': 'TXN_2026-EM-GBR-ORD#77312_GBP£3,250.75_STATUS:FLAGGED_FLAG:FRAUD',
        'feedback': 'Refund requested. Inaccurate charges on invoice. #Dispute #Chargeback'
    },
    {
        'raw_id': 'USR-105',
        'raw_name': None,  # Realistic missing data test
        'raw_contact': None,
        'raw_txn': 'TXN_2025-NA-CAN-ORD#33109_CAD$6,800.00_STATUS:COMPLETED_FLAG:NONE',
        'feedback': 'Seamless integration with existing APIs. #APIFirst #Productivity'
    }
]

df_raw = pd.DataFrame(raw_customer_records)
print("Raw Unstructured Customer Audit Dataset:")
display(df_raw)

MODULE 2.1: MESSY UNSTRUCTURED TELEMETRY & AUDIT DATA
Raw Unstructured Customer Audit Dataset:


,raw_id,raw_name,raw_contact,raw_txn,feedback
0,USR-101,"dr. alexander j. o'connor, ph.d.",+1 (800) 555-0199 ext 402,"TXN_2026-NA-USA-ORD#88412_USD$14,500.50_STATUS...",Amazing experience! Loved the #Speed and #Reli...
1,USR-102,ms. priya sharma-patel,+91 98765-43210,"TXN_2025-AP-IND-ORD#19042_INR₹4,20,000.00_STAT...",Great support team. Solved #Billing issue with...
2,USR-103,herr DR. KLAUS VON SCHMIDT,+49 (030) 123456,"TXN_2026-EM-DEU-ORD#99231_EUR€8,950.00_STATUS:...",System was down yesterday. #Downtime #Frustrat...
3,USR-104,SARAH CONNOR,020 7946 0912,"TXN_2026-EM-GBR-ORD#77312_GBP£3,250.75_STATUS:...",Refund requested. Inaccurate charges on invoic...
4,USR-105,None,None,"TXN_2025-NA-CAN-ORD#33109_CAD$6,800.00_STATUS:...",Seamless integration with existing APIs. #APIF...


### 2.2 Vectorized Cleaning, Normalization & Splitting

- **`.str.strip()`:** Strips leading/trailing whitespace.
- **`.str.replace()`:** Regular expression substitution.
- **`.str.split(..., expand=True)`:** Splits delimited text directly into individual DataFrame columns without looping!

In [6]:
print("=" * 80)
print("MODULE 2.2: VECTORIZED STRING CLEANING & EXPANSION")
print("=" * 80)

# 1. Clean honorifics, quotes, extra whitespace, and normalize casing
clean_names = (
    df_raw['raw_name']
    .str.strip()
    .str.replace(r'(?i)\b(dr|ms|herr|mr|mrs|ph\.d\.)\b\.?', '', regex=True)
    .str.replace(r'["\']', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)

print("--- [A] Cleaned and Normalized Names ---")
display(pd.DataFrame({'Raw_Name': df_raw['raw_name'], 'Clean_Name': clean_names}))

# 2. Vectorized Delimiter Split with expand=True
# Split clean name into First Name and Last Name
name_split_df = clean_names.str.split(' ', n=1, expand=True)
name_split_df.columns = ['First_Name', 'Last_Name']

print("\n--- [B] Vectorized Splitting into Structured Columns (expand=True) ---")
display(name_split_df)

MODULE 2.2: VECTORIZED STRING CLEANING & EXPANSION
--- [A] Cleaned and Normalized Names ---


,Raw_Name,Clean_Name
0,"dr. alexander j. o'connor, ph.d.","Alexander J. Oconnor, Ph.D."
1,ms. priya sharma-patel,Priya Sharma-Patel
2,herr DR. KLAUS VON SCHMIDT,Klaus Von Schmidt
3,SARAH CONNOR,Sarah Connor
4,None,None



--- [B] Vectorized Splitting into Structured Columns (expand=True) ---


,First_Name,Last_Name
0,Alexander,"J. Oconnor, Ph.D."
1,Priya,Sharma-Patel
2,Klaus,Von Schmidt
3,Sarah,Connor
4,None,None


### 2.3 Regular Expression Extraction with Named Capture Groups (`.str.extract`)

When data arrives in compound composite codes (e.g. `TXN_2026-NA-USA-ORD#88412_USD$14,500.50_STATUS:COMPLETED_FLAG:AUDIT`), writing multiple substring parses is fragile.

**Named Capture Groups (`?P<GroupName>pattern`)** allow Pandas to parse and unpack all constituent fields into structured, typed DataFrame columns in a **single compiled C pass**:

In [7]:
print("=" * 80)
print("MODULE 2.3: REGEX EXTRACTION WITH NAMED CAPTURE GROUPS")
print("=" * 80)

# Composite Regex Pattern with Named Capture Groups
txn_regex = (
    r'TXN_(?P<Year>\d{4})-'
    r'(?P<Region>[A-Z]{2})-'
    r'(?P<Country>[A-Z]{3})-'
    r'ORD#(?P<Order_ID>\d+)_'
    r'(?P<Currency>[A-Z]{3}|[^\d]+)'
    r'(?P<Raw_Amount>[\d,]+(?:\.\d{2})?)_'
    r'STATUS:(?P<Status>[A-Z]+)_'
    r'FLAG:(?P<Audit_Flag>[A-Z]+)'
)

# Extract directly into a structured DataFrame
extracted_txns = df_raw['raw_txn'].str.extract(txn_regex)

# Clean extracted monetary amount: strip currency symbols, commas, and cast to float
extracted_txns['Clean_Amount'] = (
    extracted_txns['Raw_Amount']
    .str.replace(',', '', regex=False)
    .astype(float)
)
extracted_txns['Year'] = extracted_txns['Year'].astype(int)

print("Extracted Structured Transaction Ledger:")
display(extracted_txns)

MODULE 2.3: REGEX EXTRACTION WITH NAMED CAPTURE GROUPS
Extracted Structured Transaction Ledger:


,Year,Region,Country,Order_ID,Currency,Raw_Amount,Status,Audit_Flag,Clean_Amount
0,2026,NA,USA,88412,USD$,"14,500.50",COMPLETED,AUDIT,14500.50
1,2025,AP,IND,19042,INR₹,"4,20,000.00",PENDING,NONE,420000.00
2,2026,EM,DEU,99231,EUR€,"8,950.00",COMPLETED,EXPRESS,8950.00
3,2026,EM,GBR,77312,GBP£,"3,250.75",FLAGGED,FRAUD,3250.75
4,2025,NA,CAN,33109,CAD$,"6,800.00",COMPLETED,NONE,6800.00


### 2.4 Multiple Matches (`extractall`) & Regex Backreferences

- **`.str.extractall(regex)`:** Finds **all** occurrences of a pattern within each row and outputs a MultiIndex DataFrame indexed by `(row_index, match_number)`. Ideal for extracting tags, mentions, or multi-item receipts.
- **Regex Substitution with Backreferences:** Reorders and re-formats matching tokens using `\1`, `\2` or `\g<name>` references.

In [8]:
print("=" * 80)
print("MODULE 2.4: MULTIPLE MATCHES (EXTRACTALL) & REGEX BACKREFERENCES")
print("=" * 80)

# 1. Extract all Hashtags from unstructured user feedback
print("--- [A] Extracting All Hashtags via .str.extractall() ---")
hashtag_pattern = r'#(?P<Hashtag>[A-Za-z0-9_]+)'
extracted_tags = df_raw['feedback'].str.extractall(hashtag_pattern)
display(extracted_tags)

# 2. Standardize Contact Numbers using Regex Backreferences
print("\n--- [B] Standardizing Phone Numbers via Regex Backreferences ---")
contact_series = df_raw['raw_contact'].fillna('')

# Pattern capturing country code, area code, and remaining digits
standardized_contact = (
    contact_series
    .str.replace(r'ext\s*\d+', '', regex=True)
    .str.replace(r'[^\d+]', '', regex=True)  # keep only digits and +
)

display(pd.DataFrame({
    'Raw_Contact': df_raw['raw_contact'],
    'Digits_Only': standardized_contact
}))

MODULE 2.4: MULTIPLE MATCHES (EXTRACTALL) & REGEX BACKREFERENCES
--- [A] Extracting All Hashtags via .str.extractall() ---


Hashtag
  match              
0 0             Speed
  1       Reliability
  2           TopTier
1 0           Billing
  1        SupportWin
2 0          Downtime
  1        Frustrated
3 0           Dispute
  1        Chargeback
4 0          APIFirst
  1      Productivity


--- [B] Standardizing Phone Numbers via Regex Backreferences ---


,Raw_Contact,Digits_Only
0,+1 (800) 555-0199 ext 402,+18005550199
1,+91 98765-43210,+919876543210
2,+49 (030) 123456,+49030123456
3,020 7946 0912,02079460912
4,None,


---
## 3. Transform Pipeline

### Declarative Pipeline Architecture:
Procedural data manipulation (`df['a'] = ...`, `df['b'] = ...` across 50 cells) creates:
- Hidden mutable side effects.
- Inability to unit-test individual transformations.
- Difficulty reproducing transformations in production inference pipelines.

### The Modern Solution: Method Chaining with `df.pipe()`
Each step is a pure function taking a DataFrame and returning a transformed DataFrame:

```python
clean_df = (
    raw_df
    .pipe(clean_column_names)
    .pipe(impute_missing_by_group, group_col='department')
    .pipe(standardize_group_zscores, metric_cols=['sales'])
    .pipe(winsorize_pipeline, lower=0.01, upper=0.99)
    .pipe(assert_pipeline_integrity)
)
```

### `groupby().transform()` vs. `groupby().apply()`
- **`apply()`:** Free-form aggregation or reduction. Can alter the number of rows, causing alignment issues.
- **`transform()`:** **Guarantees** output has the **exact same length and index** as the input DataFrame. Enables direct assignment of group-level aggregates (e.g. group means, group medians, group Z-scores) back to original rows without joining!

In [9]:
print("=" * 80)
print("MODULE 3.1: SYNTHETIC ENTERPRISE PIPELINE DATASET")
print("=" * 80)

# Generate realistic enterprise departmental performance dataset with intentional dirt
n_records = 20
departments = rng.choice(['Engineering', 'Marketing', 'Sales', 'Finance'], size=n_records)
base_salaries = {
    'Engineering': 110000,
    'Marketing': 75000,
    'Sales': 85000,
    'Finance': 95000
}

salaries = [base_salaries[d] + rng.normal(0, 12000) for d in departments]
sales_volume = [rng.uniform(20000, 180000) if d == 'Sales' else rng.uniform(5000, 40000) for d in departments]

df_pipeline_raw = pd.DataFrame({
    ' Employee_Name ': [f' Emp_{i:03d} ' for i in range(n_records)],
    'DEPT': departments,
    'Annual Salary ($)': np.round(salaries, 2),
    'Quarterly_Sales': np.round(sales_volume, 2),
    'Sick_Days': rng.integers(0, 15, size=n_records)
})

# Inject missing values (NaNs) and an outlier
df_pipeline_raw.loc[2, 'Annual Salary ($)'] = np.nan
df_pipeline_raw.loc[7, 'Quarterly_Sales'] = np.nan
df_pipeline_raw.loc[12, 'Annual Salary ($)'] = 850000.0  # Extreme outlier

print("Raw Dataset for Pipeline Processing:")
display(df_pipeline_raw.head(8))

MODULE 3.1: SYNTHETIC ENTERPRISE PIPELINE DATASET
Raw Dataset for Pipeline Processing:


,Employee_Name,DEPT,Annual Salary ($),Quarterly_Sales,Sick_Days
0,Emp_000,Marketing,69920.42,11187.05,9
1,Emp_001,Engineering,109043.38,34981.50,8
2,Emp_002,Engineering,NaN,31548.18,4
3,Emp_003,Engineering,92634.65,30181.20,0
4,Emp_004,Finance,79127.60,20123.26,2
5,Emp_005,Sales,73033.04,120369.41,14
6,Emp_006,Sales,89797.29,113455.68,6
7,Emp_007,Marketing,64134.25,NaN,7


In [10]:
print("=" * 80)
print("MODULE 3.2: DEFINING MODULAR PIPELINE STAGES")
print("=" * 80)

# Stage 1: Standardize Column Names to snake_case and clean strings
def clean_headers_and_strings(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Normalize column names: lower case, strip whitespace, replace symbols with underscore
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r'[\s($)]+', '_', regex=True)
        .str.strip('_')
    )
    # Strip string columns
    str_cols = df.select_dtypes(include='object').columns
    for c in str_cols:
        df[c] = df[c].str.strip()
    return df

# Stage 2: Group-Specific Median Imputation using transform()
def impute_departmental_medians(df: pd.DataFrame, group_col: str, num_cols: list) -> pd.DataFrame:
    df = df.copy()
    for col in num_cols:
        group_median = df.groupby(group_col)[col].transform('median')
        df[col] = df[col].fillna(group_median)
    return df

# Stage 3: Outlier Capping (Winsorization) per department
def winsorize_salary(df: pd.DataFrame, col: str, group_col: str) -> pd.DataFrame:
    df = df.copy()
    # Compute 95th percentile per group
    p95 = df.groupby(group_col)[col].transform(lambda s: s.quantile(0.95))
    df[col] = np.where(df[col] > p95, p95, df[col])
    return df

# Stage 4: Feature Engineering & Group-Specific Z-Scores via transform()
def engineer_metrics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Group-level Z-Score: (X - group_mean) / group_std
    dept_mean = df.groupby('dept')['annual_salary'].transform('mean')
    dept_std = df.groupby('dept')['annual_salary'].transform('std')
    df['salary_zscore_dept'] = np.round((df['annual_salary'] - dept_mean) / dept_std, 3)
    
    # Department Sales Contribution %: (Sales / Dept_Total_Sales) * 100
    dept_total_sales = df.groupby('dept')['quarterly_sales'].transform('sum')
    df['dept_sales_pct'] = np.round((df['quarterly_sales'] / dept_total_sales) * 100, 2)
    return df

# Stage 5: Schema Validation & Assertion Check
def validate_schema(df: pd.DataFrame) -> pd.DataFrame:
    assert df['annual_salary'].isna().sum() == 0, "Pipeline Error: Unresolved NaNs in annual_salary!"
    assert df['quarterly_sales'].isna().sum() == 0, "Pipeline Error: Unresolved NaNs in quarterly_sales!"
    assert (df['annual_salary'] > 0).all(), "Pipeline Error: Non-positive salary detected!"
    return df

print("Pipeline Stage Functions Defined Successfully:")
print("1. clean_headers_and_strings()")
print("2. impute_departmental_medians()")
print("3. winsorize_salary()")
print("4. engineer_metrics()")
print("5. validate_schema()")

MODULE 3.2: DEFINING MODULAR PIPELINE STAGES
Pipeline Stage Functions Defined Successfully:
1. clean_headers_and_strings()
2. impute_departmental_medians()
3. winsorize_salary()
4. engineer_metrics()
5. validate_schema()


In [11]:
print("=" * 80)
print("MODULE 3.3: EXECUTING COMPOSEABLE TRANSFORM PIPELINE VIA .pipe()")
print("=" * 80)

# Execute full declarative method-chaining pipeline
df_pipeline_clean = (
    df_pipeline_raw
    .pipe(clean_headers_and_strings)
    .pipe(impute_departmental_medians, group_col='dept', num_cols=['annual_salary', 'quarterly_sales'])
    .pipe(winsorize_salary, col='annual_salary', group_col='dept')
    .pipe(engineer_metrics)
    .pipe(validate_schema)
)

print("Pipeline Execution Completed Cleanly. Verified Zero NaNs.")
display(df_pipeline_clean.head(10))

MODULE 3.3: EXECUTING COMPOSEABLE TRANSFORM PIPELINE VIA .pipe()
Pipeline Execution Completed Cleanly. Verified Zero NaNs.


,employee_name,dept,annual_salary,quarterly_sales,sick_days,salary_zscore_dept,dept_sales_pct
0,Emp_000,Marketing,69920.420,11187.05,9,0.103,11.59
1,Emp_001,Engineering,109043.380,34981.50,8,-0.037,23.07
2,Emp_002,Engineering,109043.380,31548.18,4,-0.037,20.80
3,Emp_003,Engineering,92634.650,30181.20,0,-1.672,19.90
4,Emp_004,Finance,79127.600,20123.26,2,-0.592,46.66
5,Emp_005,Sales,73033.040,120369.41,14,-1.302,25.06
6,Emp_006,Sales,89797.290,113455.68,6,0.793,23.62
7,Emp_007,Marketing,64134.250,11187.05,7,-0.448,11.59
8,Emp_008,Sales,80462.050,33511.09,7,-0.374,6.98
9,Emp_009,Engineering,122996.325,19553.26,11,1.354,12.89


---
## 4. Memory Optimisation and Data Type Downcasting

### Memory Architecture of Pandas DataFrames:
By default, 64-bit platforms allocate:
- `int64`: 8 bytes per integer (range: $\pm 9.22 \times 10^{18}$). Storing customer ages (0–100) or months (1–12) in `int64` wastes **7 out of 8 bytes (87.5% memory bloat)**!
- `float64`: 8 bytes per float. Most analytics metrics require only 4 bytes (`float32`, precision up to 7 decimal digits).
- `object`: Stores 64-bit pointers (8 bytes) pointing to Python string wrappers scattered on the heap. This causes massive memory overhead and severe CPU cache misses.

### Downcasting Decision Matrix:
| Target DataType | Byte Width | Range / Capacity | Ideal Use Case |
| :--- | :---: | :--- | :--- |
| `int8` | 1 byte | $-128$ to $+127$ | Age, Month, Day of week, Ratings (1–5) |
| `int16` | 2 bytes | $-32,768$ to $+32,767$ | Days of year, Store IDs ($< 30k$) |
| `int32` | 4 bytes | $-2.14 \times 10^9$ to $+2.14 \times 10^9$ | Customer IDs, Order counts |
| `float32` | 4 bytes | $10^{-38}$ to $10^{38}$ (7 decimals) | Prices, Sales, Percentages, Latency |
| `category` | 1–2 bytes code | Up to $65,536$ unique categories | Strings where Unique Count $< 50\%$ of rows |

In [12]:
print("=" * 80)
print("MODULE 4.1: LARGE SCALE DATASET GENERATION (250,000 ROWS)")
print("=" * 80)

n_rows_large = 250_000

# Generate 250k rows of enterprise transaction data
df_unoptimized = pd.DataFrame({
    'transaction_id': np.arange(1, n_rows_large + 1, dtype=np.int64),
    'customer_age': rng.integers(18, 85, size=n_rows_large, dtype=np.int64),
    'store_rating': rng.integers(1, 6, size=n_rows_large, dtype=np.int64),
    'region': rng.choice(['North America', 'EMEA', 'APAC', 'LATAM'], size=n_rows_large),
    'device_type': rng.choice(['Mobile iOS', 'Mobile Android', 'Desktop Web', 'Tablet'], size=n_rows_large),
    'payment_method': rng.choice(['Credit Card', 'PayPal', 'Wire Transfer', 'Apple Pay'], size=n_rows_large),
    'order_value': rng.uniform(10.0, 1500.0, size=n_rows_large).astype(np.float64),
    'discount_pct': rng.uniform(0.0, 0.40, size=n_rows_large).astype(np.float64),
    'is_fraud_flag': rng.choice([0, 1], size=n_rows_large, p=[0.99, 0.01]).astype(np.int64)
})

# Deep Memory Profiling
baseline_mem_bytes = df_unoptimized.memory_usage(deep=True).sum()
baseline_mem_mb = baseline_mem_bytes / (1024 ** 2)

print(f"Generated {n_rows_large:,} rows.")
print(f"Baseline Unoptimized Memory Usage: {baseline_mem_mb:.2f} MB")
print("\nMemory Breakdown per Column (MB):")
mem_breakdown = (df_unoptimized.memory_usage(deep=True) / (1024 ** 2)).round(2)
display(mem_breakdown)

MODULE 4.1: LARGE SCALE DATASET GENERATION (250,000 ROWS)


Generated 250,000 rows.
Baseline Unoptimized Memory Usage: 52.81 MB

Memory Breakdown per Column (MB):


Index              0.00
transaction_id     1.91
customer_age       1.91
store_rating       1.91
region            13.23
device_type       14.13
payment_method    14.01
order_value        1.91
discount_pct       1.91
is_fraud_flag      1.91
dtype: float64

### 4.2 Automated Universal Memory Optimizer

A robust production function `optimize_dataframe_memory()` that:
1. Iterates over all numeric columns, inspecting `min()` and `max()`.
2. Selects the tightest integer bitwidth (`int8`, `int16`, `int32`, `int64`) or float bitwidth (`float32`).
3. Evaluates string cardinality ratio: $\frac{\text{Unique Count}}{\text{Total Rows}}$. If $< 50\%$, safely converts to `category`.
4. Converts binary numeric columns ($0/1$) to `bool`.

In [13]:
print("=" * 80)
print("MODULE 4.2: AUTOMATED MEMORY OPTIMIZATION FUNCTION")
print("=" * 80)

def optimize_dataframe_memory(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    # Analyzes DataFrame columns and downcasts types safely to minimize RAM footprint
    start_mem = df.memory_usage(deep=True).sum() / (1024 ** 2)
    df_opt = df.copy()

    for col in df_opt.columns:
        col_type = df_opt[col].dtype

        # 1. Handle Numeric Columns
        if pd.api.types.is_numeric_dtype(col_type):
            c_min = df_opt[col].min()
            c_max = df_opt[col].max()

            # Check for Boolean / Binary columns
            if pd.api.types.is_integer_dtype(col_type) and c_min == 0 and c_max == 1:
                df_opt[col] = df_opt[col].astype(bool)
            # Integer Downcasting
            elif pd.api.types.is_integer_dtype(col_type):
                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df_opt[col] = df_opt[col].astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df_opt[col] = df_opt[col].astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df_opt[col] = df_opt[col].astype(np.int32)
                else:
                    df_opt[col] = df_opt[col].astype(np.int64)
            # Float Downcasting
            elif pd.api.types.is_float_dtype(col_type):
                if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                    df_opt[col] = df_opt[col].astype(np.float32)

        # 2. Handle Object / String Columns -> Categorical
        elif col_type == 'object':
            n_unique = df_opt[col].nunique()
            n_total = len(df_opt[col])
            # If cardinality ratio < 50%, convert to category
            if n_unique / n_total < 0.50:
                df_opt[col] = df_opt[col].astype('category')

    end_mem = df_opt.memory_usage(deep=True).sum() / (1024 ** 2)
    reduction = ((start_mem - end_mem) / start_mem) * 100

    if verbose:
        print("=" * 60)
        print("         MEMORY OPTIMIZATION AUDIT REPORT         ")
        print("=" * 60)
        print(f"Original RAM Footprint:  {start_mem:8.2f} MB")
        print(f"Optimized RAM Footprint: {end_mem:8.2f} MB")
        print(f"RAM Reduction:           {reduction:8.2f} %")
        print("=" * 60)

    return df_opt

# Run optimization on our 250k row dataset
df_optimized = optimize_dataframe_memory(df_unoptimized, verbose=True)

# Verify mathematical data integrity
print("\nVerifying Statistical Integrity (Original vs. Optimized):")
diff_order_value = abs(df_unoptimized['order_value'].mean() - df_optimized['order_value'].mean())
diff_customer_age = abs(df_unoptimized['customer_age'].mean() - df_optimized['customer_age'].mean())
print(f"Mean Order Value Absolute Difference:  {diff_order_value:.6f}")
print(f"Mean Customer Age Absolute Difference: {diff_customer_age:.6f}")
assert diff_order_value < 1e-3, "Integrity Check Failed: Numerical drift detected!"

MODULE 4.2: AUTOMATED MEMORY OPTIMIZATION FUNCTION


         MEMORY OPTIMIZATION AUDIT REPORT         
Original RAM Footprint:     52.81 MB
Optimized RAM Footprint:     4.29 MB
RAM Reduction:              91.87 %

Verifying Statistical Integrity (Original vs. Optimized):
Mean Order Value Absolute Difference:  0.000055
Mean Customer Age Absolute Difference: 0.000000


---
## 5. High-Performance Query (`df.query` & `pd.eval`)

### The Latent Flaw in Standard Boolean Masking:
When evaluating complex boolean filters:
```python
df[(df['age'] > 30) & (df['salary'] < 100000) & (df['dept'] == 'Sales')]
```
Python executes this in multiple intermediate steps:
1. Allocates a temporary boolean array of size $N$ for `df['age'] > 30`.
2. Allocates a second temporary boolean array of size $N$ for `df['salary'] < 100000`.
3. Allocates a third temporary boolean array of size $N$ for bitwise `&`.
4. Allocates a fourth temporary boolean array for `df['dept'] == 'Sales'`.
5. Allocates a fifth array for the final combination.

On datasets with millions of rows, allocating intermediate RAM arrays creates cache misses, triggers Python garbage collection spikes, and slows down processing.

### The High-Performance Engine: `df.query()` and `pd.eval()` with NumExpr:
1. **No Temporary Arrays:** Compiles the entire expression into C-level bytecode and evaluates elements in CPU cache chunks.
2. **Multi-Threaded Execution:** NumExpr automatically divides array chunks across all available CPU cores.
3. **Syntax Elegance:** Supports `@variable` bindings and handles column names containing spaces using backticks (`` `Col Name` ``).

In [14]:
print("=" * 80)
print("MODULE 5.1: QUERY SYNTAX MASTERY & DYNAMIC VARIABLE BINDING")
print("=" * 80)

# Prepare dataframe with column names containing spaces to showcase backtick syntax
df_sample = df_optimized.head(10000).copy()
df_sample.rename(columns={
    'order_value': 'Order Value (USD)',
    'customer_age': 'Customer Age',
    'device_type': 'Device Type'
}, inplace=True)

# 1. External Variable Binding with @
min_order_threshold = 750.0
target_regions = ['North America', 'EMEA']
max_age = 45

print("--- [A] Dynamic Query with @ Variables and Backticks ---")
query_result = df_sample.query(
    "`Order Value (USD)` >= @min_order_threshold and "
    "region in @target_regions and "
    "`Customer Age` <= @max_age and "
    "`Device Type` == 'Mobile iOS'"
)

print(f"Total Matches Found: {len(query_result):,}")
display(query_result.head(5))

# 2. Accelerated Arithmetic with df.eval()
print("\n--- [B] Computational Evaluation via df.eval() ---")
# Calculate Net Realized Order Value without temporary arrays
df_sample = df_sample.eval("Net_Order_Value = `Order Value (USD)` * (1 - discount_pct)")
display(df_sample[['Order Value (USD)', 'discount_pct', 'Net_Order_Value']].head(5))

MODULE 5.1: QUERY SYNTAX MASTERY & DYNAMIC VARIABLE BINDING
--- [A] Dynamic Query with @ Variables and Backticks ---
Total Matches Found: 271


,transaction_id,Customer Age,store_rating,region,Device Type,payment_method,Order Value (USD),discount_pct,is_fraud_flag
11,12,19,3,EMEA,Mobile iOS,Wire Transfer,904.934448,0.257351,False
32,33,39,1,EMEA,Mobile iOS,PayPal,1382.072388,0.305927,False
68,69,28,2,EMEA,Mobile iOS,Apple Pay,1195.677734,0.385057,False
70,71,24,1,EMEA,Mobile iOS,Credit Card,760.202759,0.206143,False
82,83,40,4,EMEA,Mobile iOS,Credit Card,1288.838379,0.163554,False



--- [B] Computational Evaluation via df.eval() ---


,Order Value (USD),discount_pct,Net_Order_Value
0,1195.979858,0.153856,1011.971008
1,143.515579,0.127923,125.156670
2,1397.572144,0.076864,1290.148682
3,1232.742432,0.124706,1079.012451
4,588.767639,0.151777,499.406372


### 5.2 Large-Scale Performance Benchmark (1,000,000 Rows)

We generate **1,000,000 records** and run high-resolution runtime benchmarks across:
1. **Standard Boolean Indexing:** `df[(df['col1'] > ...) & (df['col2'] < ...)]`
2. **`df.query(..., engine='python')`:** Python AST evaluation
3. **`df.query(..., engine='numexpr')`:** Multi-threaded C bytecode evaluation

In [15]:
print("=" * 80)
print("MODULE 5.2: 1,000,000 ROW BENCHMARK (BOOLEAN vs QUERY PYTHON vs QUERY NUMEXPR)")
print("=" * 80)

n_benchmark = 1_000_000
print(f"Generating synthetic benchmark dataset with {n_benchmark:,} rows...")

df_large = pd.DataFrame({
    'A': rng.uniform(0.0, 100.0, size=n_benchmark).astype(np.float32),
    'B': rng.uniform(0.0, 100.0, size=n_benchmark).astype(np.float32),
    'C': rng.integers(1, 1000, size=n_benchmark, dtype=np.int32),
    'Category': rng.choice(['Alpha', 'Beta', 'Gamma', 'Delta'], size=n_benchmark)
})
df_large['Category'] = df_large['Category'].astype('category')

# Filter Condition: (A > 50.0) & (B < 30.0) & (C >= 250)
threshold_A = 50.0
threshold_B = 30.0
threshold_C = 250

n_runs = 5
times_bool = []
times_query_py = []
times_query_numexpr = []

# 1. Benchmark Standard Boolean Indexing
for _ in range(n_runs):
    t0 = time.perf_counter()
    res1 = df_large[(df_large['A'] > threshold_A) & (df_large['B'] < threshold_B) & (df_large['C'] >= threshold_C)]
    times_bool.append((time.perf_counter() - t0) * 1000)

# 2. Benchmark df.query() with Python engine
for _ in range(n_runs):
    t0 = time.perf_counter()
    res2 = df_large.query("A > @threshold_A and B < @threshold_B and C >= @threshold_C", engine='python')
    times_query_py.append((time.perf_counter() - t0) * 1000)

# 3. Benchmark df.query() with NumExpr engine
for _ in range(n_runs):
    t0 = time.perf_counter()
    res3 = df_large.query("A > @threshold_A and B < @threshold_B and C >= @threshold_C", engine='numexpr')
    times_query_numexpr.append((time.perf_counter() - t0) * 1000)

# Verify all results return exact identical rows
assert len(res1) == len(res2) == len(res3), "Benchmark Error: Result row counts mismatch!"

mean_bool = np.mean(times_bool)
mean_py = np.mean(times_query_py)
mean_numexpr = np.mean(times_query_numexpr)

print("\n" + "=" * 70)
print("       HIGH-PERFORMANCE FILTERING BENCHMARK SCORECARD       ")
print("=" * 70)
print(f"Dataset Size:           {n_benchmark:,} rows")
print(f"Matching Rows:          {len(res1):,} rows ({(len(res1)/n_benchmark)*100:.1f}%)")
print("-" * 70)
print(f"1. Standard Boolean Mask:     {mean_bool:7.2f} ms  (Std: {np.std(times_bool):.2f} ms)")
print(f"2. df.query (Python engine):  {mean_py:7.2f} ms  (Std: {np.std(times_query_py):.2f} ms)")
print(f"3. df.query (NumExpr engine): {mean_numexpr:7.2f} ms  (Std: {np.std(times_query_numexpr):.2f} ms)")
print("-" * 70)
speedup = mean_py / mean_numexpr
print(f"NumExpr vs Python Engine Speedup Factor: {speedup:.2f}x Faster")
print("=" * 70)

MODULE 5.2: 1,000,000 ROW BENCHMARK (BOOLEAN vs QUERY PYTHON vs QUERY NUMEXPR)
Generating synthetic benchmark dataset with 1,000,000 rows...



       HIGH-PERFORMANCE FILTERING BENCHMARK SCORECARD       
Dataset Size:           1,000,000 rows
Matching Rows:          112,552 rows (11.3%)
----------------------------------------------------------------------
1. Standard Boolean Mask:        9.86 ms  (Std: 3.60 ms)
2. df.query (Python engine):    12.74 ms  (Std: 1.06 ms)
3. df.query (NumExpr engine):   14.23 ms  (Std: 0.65 ms)
----------------------------------------------------------------------
NumExpr vs Python Engine Speedup Factor: 0.90x Faster


---
## 6. Summary & Architectural Scorecard

| Module | Core Tool / API | Primary Advantage | When to Use | Key Gotcha |
| :--- | :--- | :--- | :--- | :--- |
| **1. MultiIndex** | `pd.IndexSlice`, `.xs()`, `stack()` / `unstack()` | Represents $N$-dimensional data cubes in 2D with $O(1)$ lookups. | Hierarchical reports (Region $\to$ Country $\to$ Year). | Always run `df.sort_index()` before range slicing to prevent errors. |
| **2. Vectorised Strings** | `.str` accessor, Named Regex `(?P<Name>...)` | Compiled C routines; automatic `NaN` propagation; eliminates slow loops. | Parsing unstructured transaction logs, phone numbers, and tags. | Avoid `.apply(lambda x: ...)` for strings; use `.str.extract()` instead. |
| **3. Transform Pipeline** | `df.pipe()`, `groupby().transform()` | Pure functional composability; preserves input index length. | Multi-stage cleaning, group-specific Z-score scaling and imputation. | Use `transform()` instead of `apply()` when broadcasting back to rows. |
| **4. Memory Optimisation** | `pd.to_numeric(downcast=...)`, `category` dtype | Drastically cuts RAM usage ($> 80\%$) and speeds up groupbys. | Datasets with $\ge 100k$ rows or memory-constrained servers. | Ensure unique ratio $< 50\%$ before converting strings to `category`. |
| **5. High-Performance Query** | `df.query(engine='numexpr')`, `pd.eval()` | Avoids temporary RAM array allocations; multi-threaded C execution. | Complex multi-condition filtering on datasets with $> 100k$ rows. | Use backticks (`` `Col Name` ``) for column names containing spaces. |

---
**Module Completed:** Day 9 Advanced Data Wrangling & High-Performance Pandas  
**Author:** abs6187 (`23f2000876@ds.study.iitm.ac.in`)